# Per-Subject DSAR/CCPA Erasure — production job (single notebook)

One directly-runnable notebook that folds the step-by-step demo (`01`→`04`) into a single
production task: **load config → erase (honouring `request_type`) → physically purge → validate → report**.
Deploy it as a monthly Databricks Job.

**What changed vs. the demo notebooks (`01`–`04`):**

- **No demo scaffolding.** It reads an *existing* `dsar_request` table and an *existing* PII column
  registry — it does not create schemas or synthetic data. Run `00` only if you want the demo dataset.
- **Match rule = email-primary, name fallback** (Allegiant's stated rule). If a table exposes an `email`
  identifier, the subject is matched on **email alone**; only tables with *no* email column fall back to
  **first + last name**. See §3.
- **Employee/Customer scope.** DSAR/CCPA requests come from **customers**. Tables tagged as
  `employee`-scoped (e.g. Merlot crew/internal tables) are **skipped by default** so we never scan or erase
  employee data on a customer request. See §2.
- **`dry_run` guard on by default** — you must set `dry_run=false` to actually write.

> This notebook only *reads* the registry; keep tagging/registry seeding in `01_pii_column_registry`
> (tag-driven). It reuses the same registry schema plus one optional column, `subject_scope`.


## 0. Configuration

In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "dkushari_uc", "1 Catalog")
dbutils.widgets.text("schema",  "allegiant_air_dsar", "2 Schema")
dbutils.widgets.text("redaction_token", "***REDACTED***", "3 Scalar erasure token")
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"], "4 Dry run (true = count only, no writes)")
dbutils.widgets.dropdown("do_purge", "true", ["true", "false"], "5 Physically VACUUM after erase")
dbutils.widgets.text("subject_scope", "customer", "6 Scope to erase (customer | employee | all)")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")
TOKEN   = dbutils.widgets.get("redaction_token")
DRY     = dbutils.widgets.get("dry_run") == "true"
DO_PURGE = dbutils.widgets.get("do_purge") == "true"
SCOPE   = dbutils.widgets.get("subject_scope").strip().lower()
FQ      = f"{CATALOG}.{SCHEMA}"

from collections import defaultdict
print(f"Schema: {FQ} | token: {TOKEN} | dry_run: {DRY} | do_purge: {DO_PURGE} | scope: {SCOPE}")
assert SCOPE in ("customer", "employee", "all"), "subject_scope must be customer | employee | all"


## 1. Load config: pending requests + registry
Reads the PENDING privacy requests and the PII column registry. Both already exist — the registry is
seeded from UC column tags by `01_pii_column_registry`. This notebook adds no data.

In [ ]:
requests = [r.asDict() for r in spark.sql(
    f"SELECT * FROM {FQ}.dsar_request WHERE status = 'PENDING'").collect()]
reg = [r.asDict() for r in spark.sql(f"SELECT * FROM {FQ}.pii_column_registry").collect()]
print(f"{len(requests)} pending requests; {len(reg)} registered PII columns across "
      f"{len({r['table_name'] for r in reg})} tables")


## 2. Scope filter — customer vs. employee tables
DSAR/CCPA is a **customer** right, so a customer request must never scan or erase **employee**-only
sources (e.g. Merlot crew tables). Each registry row may carry a `subject_scope` (`customer` | `employee`);
rows without one default to `customer` (safe for the existing customer tables). We keep only the tables in
the selected scope, then group the registry by table.

**How a table gets its scope:** tag the *table* (not the column) with `subject_scope=employee` in UC, and
have `01` project that tag onto its registry rows — or set the column directly. Absent a tag, a table is
treated as `customer`.

In [ ]:
def scope_of(row):
    # registry may or may not carry the column yet; default to customer.
    return (row.get("subject_scope") or "customer").strip().lower()

if SCOPE == "all":
    kept = reg
else:
    kept = [r for r in reg if scope_of(r) == SCOPE]

skipped_tables = sorted({r["table_name"] for r in reg} - {r["table_name"] for r in kept})
by_table = defaultdict(list)
for r in kept:
    by_table[r["table_name"]].append(r)

print(f"scope={SCOPE}: {len(by_table)} table(s) in scope: {sorted(by_table)}")
if skipped_tables:
    print(f"skipped (out of scope): {skipped_tables}")


## 3. Match predicate — **email-primary, name fallback**
Allegiant's rule: *match on email first; only if the table has no email do we fall back to first + last name.*

- If the table has an **`email`** identifier → match on **email alone** (unique, avoids missing a subject
  whose name is spelled differently across systems).
- Else if it has **name** identifiers → match on **first AND last** (or a single `full_name`).
- **JSON payload** identifiers → the payload string must contain the subject's email (primary). Name terms
  are added only when the request has no email, mirroring the scalar rule.

This is a deliberate change from the demo's "all identifiers must match". Email-primary is both more
correct for erasure (fewer false negatives) and closer to Allegiant's current AWS logic.

In [ ]:
def sqlstr(v):
    return "'" + (v or "").replace("'", "''") + "'"

def json_mask_expr(col):
    """Reused from the masking layer: redact name key + URL firstName/lastName/email in the JSON string."""
    e = f"regexp_replace({col}, '(\"name\" *: *)\"[^\"]*\"', '$1\"***REDACTED***\"')"
    e = f"regexp_replace({e}, '([?&](firstName|lastName|email)=)[^&#\"]*', '$1***REDACTED***')"
    return e

def match_predicate(table, req):
    """Email-primary, name-fallback predicate for this subject in this table (or None if no identifiers)."""
    ids = [r for r in by_table[table] if r["is_identifier"]]
    if not ids:
        return None
    email_cols = [r for r in ids if r["identifier_role"] == "email" and r["erase_strategy"] != "json_key"]
    json_cols  = [r for r in ids if r["erase_strategy"] == "json_key"]
    name_cols  = [r for r in ids if r["identifier_role"] in ("first_name", "last_name", "full_name")]

    # 1) scalar email present -> match on email alone
    if email_cols and req.get("email"):
        return " OR ".join(f"{r['column_name']} = {sqlstr(req['email'])}" for r in email_cols)

    # 2) JSON payload -> email primary; add name terms only when no email on the request
    if json_cols:
        preds = []
        for r in json_cols:
            col = r["column_name"]
            if req.get("email"):
                preds.append(f"{col} LIKE '%' || {sqlstr(req['email'])} || '%'")
            else:
                preds.append(f"({col} LIKE '%' || {sqlstr(req['first_name'])} || '%' AND "
                             f"{col} LIKE '%' || {sqlstr(req['last_name'])} || '%')")
        return " AND ".join(preds)

    # 3) no email column anywhere -> fall back to name (first AND last, or full_name)
    preds = []
    for r in name_cols:
        role, col = r["identifier_role"], r["column_name"]
        if role == "first_name":  preds.append(f"{col} = {sqlstr(req['first_name'])}")
        elif role == "last_name": preds.append(f"{col} = {sqlstr(req['last_name'])}")
        elif role == "full_name": preds.append(f"{col} = {sqlstr((req['first_name'] or '') + ' ' + (req['last_name'] or ''))}")
    return " AND ".join(preds) if preds else None

if requests and by_table:
    t0 = next(iter(by_table))
    print(f"example predicate ({t0}):", match_predicate(t0, requests[0]))


## 4. SET clause — what to erase once a row matches
Erase **every** PII column registered for the table (identifiers AND non-identifier PII like `pnr`):
scalars → redaction token; JSON → the compiled in-place redaction.

In [ ]:
def set_clause(table):
    sets = []
    for r in by_table[table]:
        col, strat = r["column_name"], r["erase_strategy"]
        if strat in ("scalar_token", "pnr_token"):
            sets.append(f"{col} = {sqlstr(TOKEN)}")
        elif strat == "json_key":
            sets.append(f"{col} = {json_mask_expr(col)}")
    return ", ".join(sets)


## 5. Erase — honour `request_type`, guarded by `dry_run`
- **OBFUSCATE** → `UPDATE … SET <redact PII cols> WHERE <match>` (keep the row, blank only PII cells).
- **DELETE** → `DELETE FROM … WHERE <match>` (remove the whole matched row).

With `dry_run=true` we only **count** matches and write nothing — always confirm the counts before a real run.

In [ ]:
erased_subjects = [dict(r) for r in requests]  # snapshot before we scrub the request table
audit = []
for req in requests:
    mode = (req.get("request_type") or "OBFUSCATE").upper()
    for table in by_table:
        pred = match_predicate(table, req)
        if not pred:
            continue
        fqt = f"{FQ}.{table}"
        n = spark.sql(f"SELECT count(*) c FROM {fqt} WHERE {pred}").collect()[0]["c"]
        if n and not DRY:
            if mode == "DELETE":
                spark.sql(f"DELETE FROM {fqt} WHERE {pred}")
            else:
                spark.sql(f"UPDATE {fqt} SET {set_clause(table)} WHERE {pred}")
        action = "dry_run" if DRY else ("deleted" if mode == "DELETE" else ("obfuscated" if n else "no_match"))
        audit.append((req["request_id"], req.get("email"), mode, table, n, action))

import pandas as pd
adf = pd.DataFrame(audit, columns=["request_id","email","request_type","table","rows_matched","action"])
print(("would affect" if DRY else "affected"), int(adf["rows_matched"].sum()), "rows")
display(spark.createDataFrame(adf) if len(adf) else spark.createDataFrame([], "request_id string"))


## 6. Physical purge (CCPA "no trace")
`REORG … APPLY (PURGE)` + zero-retention `VACUUM` remove the pre-erasure raw bytes so nothing is
recoverable via time-travel. Skipped entirely on a dry run. Also scrubs the request table's own raw
identifiers and marks the processed requests `COMPLETE`.

> Zero-retention VACUUM is **destructive** and disables time-travel for the table — that is the point for
> CCPA. On serverless we set `delta.deletedFileRetentionDuration = 'interval 0 hours'` then run a plain
> `VACUUM` (the retention-check session conf isn't settable on serverless/Spark Connect).

In [ ]:
def purge_table(fqt, do_vacuum=True):
    spark.sql(f"ALTER TABLE {fqt} SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours')")
    try:
        spark.sql(f"REORG TABLE {fqt} APPLY (PURGE)")
    except Exception as e:
        print("REORG skipped", fqt, str(e).splitlines()[0])
    if do_vacuum:
        spark.sql(f"VACUUM {fqt}")   # no RETAIN clause -> honours the 0-hour property

if DRY:
    print("dry_run=true -> skipping purge and request-table scrub")
else:
    # scrub the request table's raw identifiers first, then purge every in-scope table + the request table
    spark.sql(f"""UPDATE {FQ}.dsar_request
        SET first_name={sqlstr(TOKEN)}, last_name={sqlstr(TOKEN)}, email={sqlstr(TOKEN)}, status='COMPLETE'
        WHERE status='PENDING'""")
    for t in list(by_table.keys()) + ["dsar_request"]:
        purge_table(f"{FQ}.{t}", DO_PURGE)
    print("purge + request scrub done" if DO_PURGE else "REORG done (VACUUM skipped: do_purge=false)")


## 7. Validate — no residual trace
Scan every in-scope PII column for any of the processed subjects' identifiers. On a real run this should
find **nothing**; the empty result is your per-run compliance evidence. (On a dry run this simply reports
current matches, since nothing was erased.)

In [ ]:
findings = []
for subj in erased_subjects:
    needles = {"email": subj.get("email"), "first": subj.get("first_name"),
               "last": subj.get("last_name"),
               "full": (subj.get("first_name") or "") + " " + (subj.get("last_name") or "")}
    for table, cols in by_table.items():
        fqt = f"{FQ}.{table}"
        for r in cols:
            col = r["column_name"]
            for label, val in needles.items():
                if not val or not val.strip():
                    continue
                hits = spark.sql(
                    f"SELECT count(*) c FROM {fqt} WHERE {col} LIKE '%' || {sqlstr(val)} || '%'"
                ).collect()[0]["c"]
                if hits:
                    findings.append((subj["request_id"], table, col, label, hits))

import pandas as pd
if DRY:
    print("dry_run=true -> validation reflects current (un-erased) state; expect matches here.")
if findings:
    print(("dry_run matches" if DRY else "❌ VALIDATION FAILED — residual traces:"))
    display(spark.createDataFrame(pd.DataFrame(findings,
        columns=["request_id","table","column","matched_field","rows"])))
elif not DRY:
    print("✅ VALIDATION PASSED — no trace of any processed subject remains in any in-scope table.")


## 8. Report
Per-(request, table) audit and the request table's final state. In a scheduled Job, capture this cell's
output (or write `adf` to an audit table) as the run's compliance record.

In [ ]:
print("Per-(request, table) rows affected:")
display(spark.createDataFrame(adf) if len(adf) else spark.createDataFrame([], "request_id string"))
print("Request table state:")
display(spark.sql(f"SELECT request_id, request_type, request_date, deadline_date, status FROM {FQ}.dsar_request ORDER BY request_id"))
